# 14 · Calidad de los datos sintéticos

Mide si las muestras generadas se parecen a las reales, con independencia de que ayuden o no al modelo downstream.

**Entradas**

- `data/processed/ventanas.npz`
- `data/synthetic/*.npz`

**Salidas**

- `results/metricas/calidad_sintetica.csv`
- `results/metricas/momentos_sinteticos.csv`
- `results/figures/calidad_*.png`

**Tiempo estimado:** ~10 min en CPU (un boosting por generador sobre el espacio proyectado).

In [ ]:
import sys; sys.path.insert(0, "..")   # permite ejecutar desde notebooks/
import src                              # fija el backend de Keras a PyTorch
from src import config, viz
config.fijar_semillas()
viz.aplicar_estilo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import ventanas

part = ventanas.cargar_procesado()
train, val, test = part.train, part.val, part.test
print(train, val, test, sep="\n")

In [ ]:
from src import evaluacion, ventanas
from src.generadores.base import REGISTRO, GeneradorSintetico

n_regimenes = config.n_regimenes()
bloque_train = ventanas.empaquetar(train)

disponibles = [n for n in sorted(REGISTRO) if (src.DIR_SINTETICO / (n + ".npz")).exists()]
bancos = {n: GeneradorSintetico.importar_muestras(n) for n in disponibles}

print("Generadores evaluados:", disponibles)
print("bloque real de train:", bloque_train.shape)

## Qué mide cada métrica y por qué hacen falta todas

Un generador puede mejorar el downstream por pura regularización mientras produce
muestras irreconocibles, y conviene saberlo. Este notebook es independiente del
barrido: mide fidelidad, no utilidad.

**Puntuación discriminativa.** Un clasificador intenta separar reales de sintéticos.
Un AUC de 0.5 significa que no lo consigue y las distribuciones son
indistinguibles; cerca de 1, el generador deja una firma evidente. Referencia de
lectura: por debajo de 0.60 el generador es bueno, entre 0.60 y 0.75 aceptable, por
encima de 0.85 hay que rechazarlo.

**Distancia al vecino más cercano.** Test de memorización, y es una condición de
validez, no una métrica más: si el generador copia, cualquier mejora que produzca
aguas abajo es un artefacto y el resto de resultados no significan nada. El
cociente contra la distancia típica entre reales debe rondar 1.

**Error de correlaciones.** Diferencia relativa entre las matrices de correlación
**entre canales** (20 × 20), no la del bloque aplanado completo: esa última tendría
del orden de 720.000 entradas estimadas con unos miles de muestras y su norma
mediría el error de estimación más que el del generador.

**Error de autocorrelación.** La métrica que de verdad separa a los generadores.
Los retornos casi no tienen autocorrelación y reproducir eso es fácil; su valor
absoluto sí la tiene y decae lentamente —el agrupamiento de volatilidad—, y ahí es
donde un generador que solo aprendió medias y covarianzas se delata.

**Momentos.** La fila que importa es la curtosis: los retornos financieros tienen
colas gruesas y un generador gaussiano no puede reproducirlas por construcción. Esta
tabla lo demuestra en vez de afirmarlo.

Las métricas de distancia se calculan sobre una proyección PCA ajustada solo con los
reales. En 1.201 dimensiones con unos miles de muestras todas las distancias entre
pares convergen al mismo valor y el vecino más cercano deja de discriminar.

In [ ]:
N_EVALUACION = 1000
v = config.ventanas()
n_canales = config.n_canales()
rng = np.random.default_rng(config.semilla())

filas = []
for nombre in disponibles:
    bloques_sint, y_sint = bancos[nombre]
    reales = bloque_train[rng.choice(len(bloque_train), min(N_EVALUACION, len(bloque_train)), replace=False)]
    sinteticos = bloques_sint[rng.choice(len(bloques_sint), min(N_EVALUACION, len(bloques_sint)), replace=False)]

    fila = {"generador": nombre, "ambito": "global"}
    fila.update(evaluacion.bateria_calidad(
        reales, sinteticos, v.pasado, n_canales, semilla=config.semilla()
    ))
    filas.append(fila)
    print(nombre, "listo")

calidad_global = pd.DataFrame(filas)
calidad_global.round(4)

## La misma batería sobre el régimen de crisis

El agregado global está dominado por la clase mayoritaria. La pregunta del taller es
si el generador reproduce bien la clase rara, que es de la que se van a pedir
muestras adicionales, así que hay que medirla por separado.

In [ ]:
CRISIS = n_regimenes - 1
reales_crisis = bloque_train[train.y_reg == CRISIS]

filas_crisis = []
for nombre in disponibles:
    bloques_sint, y_sint = bancos[nombre]
    sinteticos_crisis = bloques_sint[y_sint == CRISIS]
    if len(sinteticos_crisis) < 50 or len(reales_crisis) < 50:
        print("Muestras insuficientes de crisis para", nombre)
        continue

    n = min(N_EVALUACION, len(reales_crisis), len(sinteticos_crisis))
    fila = {"generador": nombre, "ambito": "crisis"}
    fila.update(evaluacion.bateria_calidad(
        reales_crisis[rng.choice(len(reales_crisis), n, replace=False)],
        sinteticos_crisis[rng.choice(len(sinteticos_crisis), n, replace=False)],
        v.pasado, n_canales, semilla=config.semilla(),
    ))
    filas_crisis.append(fila)

calidad = evaluacion.acumular(filas + filas_crisis, "calidad_sintetica")
calidad.round(4)

## Momentos marginales

Una tabla por generador con los cuatro primeros momentos frente a los reales. Es la
evidencia directa de la limitación estructural del gaussiano: su diferencia de
curtosis debe ser grande y negativa, porque no puede producir colas gruesas.

In [ ]:
momentos = pd.concat(
    {
        nombre: evaluacion.comparar_momentos(bloque_train, bancos[nombre][0])["diferencia"]
        for nombre in disponibles
    },
    axis=1,
).T
momentos.index.name = "generador"

momentos.to_csv(src.DIR_METRICAS / "momentos_sinteticos.csv")
momentos.round(4)

## Fidelidad, memorización y estructura

Los fallos son opuestos y hay que mirarlos juntos: un generador puede parecer
excelente en la puntuación discriminativa precisamente porque copia. La zona buena
del primer panel es la parte baja con cociente DVMC próximo o superior a 1.

Los dos paneles de barras ordenan a los generadores por lo que peor se le da a un
modelo que solo ha aprendido momentos y covarianzas: conservar la correlación entre
canales y el agrupamiento de volatilidad.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.6))
globales = calidad[calidad["ambito"] == "global"].set_index("generador")

for nombre in globales.index:
    x = globales.loc[nombre, "cociente_dvmc"]
    y = globales.loc[nombre, "distancia_a_indistinguible"]
    ejes[0].scatter(x, y, s=90, color=viz.color(nombre), label=nombre.replace("_", " "))
    ejes[0].annotate(nombre.replace("_", " "), xy=(x, y), xytext=(7, 0),
                     textcoords="offset points", fontsize=8,
                     color=viz.TINTA_SECUNDARIA, va="center")

ejes[0].axvline(1.0, color=viz.TINTA_SECUNDARIA, linestyle="--", linewidth=1.2)
ejes[0].set_xlabel("cociente DVMC (< 1 indica memorización)")
ejes[0].set_ylabel("|AUC − 0.5|")
ejes[0].set_title("Fidelidad frente a memorización")

for eje, columna, titulo in [
    (ejes[1], "error_correlaciones", "Correlación entre canales"),
    (ejes[2], "error_acf_absolutos", "Agrupamiento de volatilidad"),
]:
    orden = globales[columna].sort_values()
    eje.barh(range(len(orden)), orden.to_numpy(), color=[viz.color(n) for n in orden.index])
    eje.set_yticks(range(len(orden)))
    eje.set_yticklabels([n.replace("_", " ") for n in orden.index])
    eje.set_xlabel(columna.replace("_", " "))
    eje.set_title(titulo)

fig.tight_layout()
viz.guardar(fig, "calidad_resumen")

## Inspección visual conjunta

Panel PCA de todos los generadores sobre el régimen de crisis, con la proyección
ajustada solo con los reales. Sirve para leer las métricas anteriores: un AUC alto
casi siempre tiene una explicación visible aquí, sea colapso de modo o dispersión
excesiva.

In [ ]:
n = len(disponibles)
filas_panel = (n + 1) // 2
fig, ejes = plt.subplots(filas_panel, 2, figsize=(12, 4.2 * filas_panel))
ejes_planos = np.ravel(ejes)

for eje, nombre in zip(ejes_planos, disponibles):
    bloques_sint, y_sint = bancos[nombre]
    sinteticos_crisis = bloques_sint[y_sint == CRISIS][:600]
    viz.real_vs_sintetico(reales_crisis, sinteticos_crisis, nombre.replace("_", " "), eje=eje)
for eje in ejes_planos[n:]:
    eje.set_visible(False)

fig.tight_layout()
viz.guardar(fig, "calidad_pca_crisis")

## Salidas generadas

In [ ]:
from pathlib import Path

salidas = [
    src.DIR_METRICAS / "calidad_sintetica.csv",
    src.DIR_METRICAS / "momentos_sinteticos.csv",
    src.DIR_FIGURAS / "calidad_resumen.png",
    src.DIR_FIGURAS / "calidad_pca_crisis.png",
]

for ruta in salidas:
    ruta = Path(ruta)
    marca = "ok" if ruta.exists() else "--"
    print("[{}] {}".format(marca, ruta.relative_to(src.RAIZ)))
